# Rate Limiting, Plainly — the simulations behind Part 6

Companion notebook for **"System Design, Plainly — Part 6: Rate Limiting"**
(2026-11-12). Every number quoted in the post is recomputed here.
Run top to bottom; each cell asserts its headline result.


## 1. The boundary burst: fixed window vs sliding window log

Limit: 100 requests per minute. An adversary sends 100 requests in the
last second of minute 1 and 100 more in the first second of minute 2.


In [ ]:
import numpy as np

LIMIT, WINDOW = 100, 60.0

def fixed_window(times, limit=LIMIT, window=WINDOW):
    admitted, counts = [], {}
    for t in times:
        w = int(t // window)
        if counts.get(w, 0) < limit:
            counts[w] = counts.get(w, 0) + 1
            admitted.append(True)
        else:
            admitted.append(False)
    return np.array(admitted)

def sliding_log(times, limit=LIMIT, window=WINDOW):
    admitted, kept = [], []
    for t in times:
        kept = [x for x in kept if x > t - window]
        if len(kept) < limit:
            kept.append(t); admitted.append(True)
        else:
            admitted.append(False)
    return np.array(admitted)

rng = np.random.default_rng(7)
times = np.sort(np.concatenate([
    rng.uniform(59.0, 59.9, 100),   # last second of minute 1
    rng.uniform(60.0, 60.9, 100),   # first second of minute 2
]))
fw, sl = fixed_window(times), sliding_log(times)
print(f"Fixed window admits {fw.sum()}/200 in 2 seconds")
print(f"Sliding log   admits {sl.sum()}/200 in 2 seconds")
assert fw.sum() == 200 and sl.sum() == 100
# Rolling 60-second admitted count, worst case over the next minute:
grid = np.arange(60.0, 120.5, 0.5)
fw_max = max(fw[(times > g - 60) & (times <= g)].sum() for g in grid)
sl_max = max(sl[(times > g - 60) & (times <= g)].sum() for g in grid)
print(f"Max requests in any rolling 60 s: fixed {fw_max}, sliding log {sl_max}")
assert fw_max == 200 and sl_max == 100


## 2. Token bucket: burst now, trickle later

Capacity 500, refill 100 tokens/s — the Stripe shape (capacity = 5x the
per-second rate). A 500-request burst at t=0, a second 500-request burst
at t=2 s, then steady 100/s traffic for 10 s.


In [ ]:
def token_bucket(arrivals, cap=500, rate=100.0):
    tokens, last = float(cap), arrivals[0]
    admitted = []
    for t in arrivals:
        tokens = min(cap, tokens + rate * (t - last))
        last = t
        if tokens >= 1:
            tokens -= 1; admitted.append(True)
        else:
            admitted.append(False)
    return np.array(admitted)

arr = np.concatenate([
    np.full(500, 0.0),                 # burst 1
    np.linspace(2.0, 2.05, 500),       # burst 2, 50 ms later in wall time
    np.arange(3.0, 13.0, 0.01),        # steady 100/s for 10 s
])
adm = token_bucket(arr)
b1, b2, st = adm[:500].sum(), adm[500:1000].sum(), adm[1000:].sum()
print(f"Burst 1 admitted {b1}/500, burst 2 admitted {b2}/500, steady {st}/1000")
assert b1 == 500 and b2 == 205 and st == 1000
# Why 205? Two seconds of refill = 200 tokens, plus 5 more refilled
# during the 50 ms the second burst takes to arrive.
print("Refill in 2 s:", 100 * 2, "tokens; during burst 2:", 100 * 0.05)


## 3. Sliding window counter vs the exact log

Cloudflare's two-counter approximation: `estimate = prev * overlap + cur`.
Poisson traffic at 90/min with a 2x spike in minutes 2-3; compare the
estimate against the true rolling count every second.


In [ ]:
rng = np.random.default_rng(11)
lam = 90.0 / 60.0
arrivals, t = [], 0.0
while t < 300:
    rate = lam * (2.0 if 120 <= t < 180 else 1.0)
    t += rng.exponential(1.0 / rate)
    if t < 300:
        arrivals.append(t)
arrivals = np.array(arrivals)

def true_rolling(t_now, arr, window=60.0):
    return ((arr > t_now - window) & (arr <= t_now)).sum()

def swc_estimate(t_now, arr, window=60.0):
    w = int(t_now // window)
    cur = ((arr >= w * window) & (arr <= t_now)).sum()
    prev = ((arr >= (w - 1) * window) & (arr < w * window)).sum()
    return prev * (window - (t_now - w * window)) / window + cur

grid = np.arange(61, 300, 1.0)
true_c = np.array([true_rolling(g, arrivals) for g in grid])
est_c = np.array([swc_estimate(g, arrivals) for g in grid])
err = np.abs(true_c - est_c)
flips = int(np.sum((true_c <= 100) != (est_c <= 100)))
print(f"Mean abs error {err.mean():.2f} reqs, max {err.max():.0f}, "
      f"mean relative error {100*err.mean()/true_c.mean():.1f}%")
print(f"Allow/reject decisions flipped at limit 100: {flips}/{len(grid)}")
assert err.mean() < 5 and flips <= 2
# Cloudflare's own measurement, 400M requests / 270k sources: 0.003% of
# decisions wrong. Our toy sim lands in the same regime: ~0.4% flipped.


## 4. GCRA: one timestamp enforces a perfectly spaced stream

Emission interval T = 10 ms (100/s), tolerance tau = 100 ms. State per key:
a single number, the Theoretical Arrival Time. A 20-request burst at t=0,
then 200/s for 0.5 s, then 50/s for 2 s.


In [ ]:
def gcra(arrivals, T=0.010, tau=0.100):
    tat, admitted = None, []
    for t in arrivals:
        if tat is None:
            tat = t
        if t >= tat - tau:
            admitted.append(True)
            tat = max(t, tat) + T
        else:
            admitted.append(False)
    return np.array(admitted)

arr = np.concatenate([
    np.full(20, 0.0),
    np.arange(0.01, 0.5, 0.005),   # 200/s
    np.arange(0.5, 2.5, 0.02),     # 50/s
])
g = gcra(arr)
n1, n2, n3 = len(arr[:20]), len(arr[20:118]), len(arr[118:])
print(f"Burst admitted {g[:20].sum()}/20 (tau/T + 1 = {int(0.100/0.010)+1})")
print(f"200/s phase admitted {g[20:118].sum()}/{n2} (~half: the cap is 100/s)")
print(f"50/s phase  admitted {g[118:].sum()}/{n3} (under the cap, ~all pass)")
assert g[:20].sum() == 11 and g[20:118].sum() == 49 and g[118:].sum() == 99


## Takeaway

- Fixed window: 1 integer of state, but a 2x burst at the boundary.
- Sliding window counter: 2 integers, ~2% error, ~0.4% flipped decisions.
- Token bucket: 2 numbers, bursts up to capacity, steady rate after.
- GCRA: 1 timestamp, zero burst beyond tau, perfectly spaced output.

Pick the algorithm by what you are protecting: the burst, the boundary,
or the downstream's steady-state rate.
